In [ ]:
# @title Analyseur de Mots de Passe par IA
# @markdown ## Projet personnel — Cybersécurité & Machine Learning
# @markdown **GitHub :** github.com/abderemaneattoumani

In [ ]:
# @title Installation des dépendances
!pip install zxcvbn reportlab -q
import math, re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from zxcvbn import zxcvbn
print("Tout est installé !")

In [ ]:
# @title Testez votre mot de passe !
mot_de_passe = "monmotdepasse123"  # @param {type:"string"}

def analyser(password):
    length = len(password)
    N = 0
    if re.search(r'[a-z]', password): N += 26
    if re.search(r'[A-Z]', password): N += 26
    if re.search(r'[0-9]', password): N += 10
    if re.search(r'[^a-zA-Z0-9]', password): N += 32
    N = max(N, 1)
    entropy = round(math.log2(N) * length, 2)
    result = zxcvbn(password)
    score = result['score']
    crack = result['crack_times_display']['offline_fast_hashing_1e10_per_second']
    
    niveaux = {0: 'Très faible', 1: 'Faible', 
               2: 'Moyen', 3: 'Fort', 4: 'Très fort'}
    
    print("=" * 50)
    print(f"  Mot de passe    : {'*' * length} ({length} caractères)")
    print(f"  Niveau          : {niveaux[score]}")
    print(f"  Entropie        : {entropy} bits")
    print(f"  Temps de crack  : {crack}")
    print(f"  Majuscules      : {sum(1 for c in password if c.isupper())}")
    print(f"  Chiffres        : {sum(1 for c in password if c.isdigit())}")
    print(f"  Caractères spéc : {sum(1 for c in password if not c.isalnum())}")
    print("=" * 50)
    
    conseils = []
    if length < 12: conseils.append("→ Allonge à 12+ caractères")
    if not re.search(r'[A-Z]', password): conseils.append("→ Ajoute des majuscules")
    if not re.search(r'[^a-zA-Z0-9]', password): conseils.append("→ Ajoute des caractères spéciaux (!@#)")
    if re.search(r'(123|abc|qwerty|azerty)', password.lower()): conseils.append("→ Évite les séquences prévisibles")
    
    if conseils:
        print("\nRecommandations :")
        for c in conseils: print(f"  {c}")
    else:
        print("\nExcellent mot de passe !")

analyser(mot_de_passe)

In [ ]:
# @title Comparaison de mots de passe
exemples = [
    "123456", "password", "jean1990",
    "Soleil12!", "X#k9mP2@qL", "correct-horse-battery-staple"
]

data = []
for pwd in exemples:
    r = zxcvbn(pwd)
    N = sum([26 if re.search(r'[a-z]',pwd) else 0,
             26 if re.search(r'[A-Z]',pwd) else 0,
             10 if re.search(r'[0-9]',pwd) else 0,
             32 if re.search(r'[^a-zA-Z0-9]',pwd) else 0]) or 1
    data.append({
        'Mot de passe': f'{pwd[:3]}{"*"*(len(pwd)-3)} ({len(pwd)}c)',
        'Score': r['score'],
        'Entropie': round(math.log2(N)*len(pwd), 1),
        'Temps crack': r['crack_times_display']['offline_fast_hashing_1e10_per_second']
    })

df = pd.DataFrame(data)
colors_bar = ['#e74c3c','#e74c3c','#e67e22','#f1c40f','#2ecc71','#2ecc71']

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(df['Mot de passe'], df['Entropie'], color=colors_bar)
ax.set_xlabel('Entropie (bits)')
ax.set_title('Comparaison de robustesse par entropie', fontweight='bold')
for bar, val in zip(bars, df['Entropie']):
    ax.text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2,
            f'{val} bits', va='center', fontsize=9)
plt.tight_layout()
plt.show()
print(df[['Mot de passe','Score','Temps crack']].to_string(index=False))